In [6]:
!cd ..

In [8]:
import os, urllib.request, urllib.parse

seq1="ATTCCGAAGTG"
seq2="TACGCAATCGA"
seq3="CGTGCTTACAG"
seq4="GTAATCTGAGC"
seq5="TGATCAGGCTA"
seq6="GACCTAAGGAT"
seq7="TCGATATGCCG"

SEQS_TO_COMPARE_TO=[seq1, seq2, seq3, seq4, seq5, seq6, seq7]

FASTA_FILE = "/content/GCF_029882485.1_ASM2988248v1_genomic.fna"

def parse_fasta(text):
    """Returns (header, sequence) of the first record in a FASTA text."""
    records = [r for r in text.strip().split(">") if r.strip()]
    if not records:
        raise ValueError("Not a FASTA file")
    if len(records) > 1:
        print(f"Note: {len(records)} records in the file, using the first one")
    lines = records[0].strip().splitlines()
    return lines[0].strip(), "".join(l.strip() for l in lines[1:]).upper()


if os.path.exists(FASTA_FILE):
    with open(FASTA_FILE) as f:
        HEADER, SEQ = parse_fasta(f.read())
    print(f"Read {FASTA_FILE}")

L = len(SEQ)
print(HEADER)
print("First 60 nt:", SEQ[:60])



Read /content/GCF_029882485.1_ASM2988248v1_genomic.fna
NC_076969.1 Acanthamoeba castellanii medusavirus DNA, complete sequence
First 60 nt: AGACGGCCTGGCAGAACTGCTCGTACACCAGCGATGGTCGGATGTGCTGGATGCGGAATC


In [10]:
import pandas as pd

ALPHABET = "ACGT"

def count_matrix(seqs, alphabet=ALPHABET):
    """Rows = letters, columns = positions (1-based). Each cell = how many
    sequences have that letter at that position."""
    length = max(len(s) for s in seqs)
    counts = {letter: [0] * length for letter in alphabet}
    for s in seqs:
        for pos, letter in enumerate(s.upper()):
            if letter not in counts:
                raise ValueError(f"Unexpected letter '{letter}' in {s}")
            counts[letter][pos] += 1
    return pd.DataFrame.from_dict(counts, orient="index",
                                  columns=range(1, length + 1))

def probability_matrix(counts, n_seqs):
    """Same shape as counts; each cell = appearances / number of sequences."""
    return counts / n_seqs


counts = count_matrix(SEQS_TO_COMPARE_TO)
probs = probability_matrix(counts, len(SEQS_TO_COMPARE_TO))

print("Count matrix:")
print(counts)
print("Probability matrix:")
print(probs.round(3))

import numpy as np

def log_odds_matrix(probs, background=0.25):
    """ln(probability / background) for each letter and position.
    Cells with probability 0 become -inf."""
    with np.errstate(divide="ignore"):
        return np.log(probs / background)


def score_windows(seq, log_mat, inf_penalty=-20):
    """Scores every window of length = number of columns in log_mat.
    scores[i] is the score of the window seq[i : i + width].
    -inf cells count as inf_penalty. Assumes seq contains only letters in log_mat."""
    letters = list(log_mat.index)                # ['A', 'C', 'G', 'T']
    width = log_mat.shape[1]                     # 11
    M = log_mat.replace(-np.inf, inf_penalty).to_numpy()   # shape (4, width)

    # Turn the sequence into row indices: A->0, C->1, G->2, T->3
    lookup = np.zeros(256, dtype=np.int64)
    for i, letter in enumerate(letters):
        lookup[ord(letter)] = i
    encoded = lookup[np.frombuffer(seq.encode("ascii"), dtype=np.uint8)]

    n_windows = len(seq) - width + 1
    scores = np.zeros(n_windows)
    for pos in range(width):
        scores += M[encoded[pos:pos + n_windows], pos]
    return scores


log_mat = log_odds_matrix(probs)
display(log_mat.round(3))

scores = score_windows(SEQ, log_mat)
print(f"Scored {len(scores):,} windows")

# Best-scoring windows
width = log_mat.shape[1]
top = np.argsort(scores)[::-1][:10]
for i in top:
    print(f"pos {i + 1:>10,}  {SEQ[i:i + width]}  score = {scores[i]:.3f}")

Count matrix:
   1   2   3   4   5   6   7   8   9   10  11
A   1   2   2   2   0   4   3   2   1   2   2
C   1   1   2   2   4   1   0   0   4   1   1
G   2   2   1   2   0   1   1   4   2   2   3
T   3   2   2   1   3   1   3   1   0   2   1
Probability matrix:
      1      2      3      4      5      6      7      8      9      10     11
A  0.143  0.286  0.286  0.286  0.000  0.571  0.429  0.286  0.143  0.286  0.286
C  0.143  0.143  0.286  0.286  0.571  0.143  0.000  0.000  0.571  0.143  0.143
G  0.286  0.286  0.143  0.286  0.000  0.143  0.143  0.571  0.286  0.286  0.429
T  0.429  0.286  0.286  0.143  0.429  0.143  0.429  0.143  0.000  0.286  0.143


,1,2,3,4,5,6,7,8,9,10,11
A,-0.560,0.134,0.134,0.134,-inf,0.827,0.539,0.134,-0.560,0.134,0.134
C,-0.560,-0.560,0.134,0.134,0.827,-0.560,-inf,-inf,0.827,-0.560,-0.560
G,0.134,0.134,-0.560,0.134,-inf,-0.560,-0.560,0.827,0.134,0.134,0.539
T,0.539,0.134,0.134,-0.560,0.539,-0.560,0.539,-0.560,-inf,0.134,-0.560


Scored 381,267 windows
pos     64,548  TACCCAAGCTG  score = 5.458
pos    379,138  TGTGCAAGCGG  score = 5.458
pos     63,250  TTTGCATGCGG  score = 5.458
pos     30,115  TGCCCATGCTG  score = 5.458
pos    330,310  TGCGCAAGCTG  score = 5.458
pos     33,889  TGCGCAAGCTG  score = 5.458
pos    137,096  TTCGCAAGCAG  score = 5.458
pos    173,717  TAACCATGCAG  score = 5.458
pos    288,003  TTTACAAGCGG  score = 5.458
pos     43,616  TGCGCATGCAG  score = 5.458
